In [ ]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


# -----------------------------------------
# 1. Load configuration from .env
# -----------------------------------------
BASE_DIR = Path(__file__).resolve().parent
ENV_FILE = BASE_DIR / ".env"

load_dotenv(ENV_FILE)


# -----------------------------------------
# 2. Read and validate configuration
# -----------------------------------------
def get_required_env(name: str) -> str:
    value = os.getenv(name)

    if value is None or value.strip() == "":
        raise RuntimeError(
            f"Missing required configuration value: {name}"
        )

    return value.strip()


DB_USER = get_required_env("DB_USER")
DB_PASSWORD = get_required_env("DB_PASSWORD")
DB_HOST = get_required_env("DB_HOST")
DB_PORT = int(get_required_env("DB_PORT"))
DATABASE_NAME = get_required_env("DATABASE_NAME")

SOURCE_SCHEMA = get_required_env("SOURCE_SCHEMA")
CLEAN_SCHEMA = get_required_env("CLEAN_SCHEMA")


# -----------------------------------------
# 3. Build PostgreSQL connection URLs
# -----------------------------------------
def make_postgres_url(database_name: str) -> URL:
    return URL.create(
        drivername="postgresql+psycopg2",
        username=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT,
        database=database_name,
    )


# Connect to the default PostgreSQL database
# before checking or creating "transport".
admin_url = make_postgres_url("postgres")

# Connection URL for the target database
database_url = make_postgres_url(DATABASE_NAME)


# -----------------------------------------
# 4. Check and create target database
# -----------------------------------------
def ensure_database_exists() -> None:
    admin_engine = create_engine(
        admin_url,
        isolation_level="AUTOCOMMIT",
    )

    try:
        with admin_engine.connect() as conn:
            database_exists = conn.execute(
                text("""
                    SELECT EXISTS (
                        SELECT 1
                        FROM pg_database
                        WHERE datname = :database_name
                    )
                """),
                {"database_name": DATABASE_NAME},
            ).scalar()

            if database_exists:
                print(
                    f"Database already exists: "
                    f"{DATABASE_NAME}"
                )
            else:
                # Database names cannot be passed as normal SQL
                # parameters. This value comes from your own .env file.
                quoted_database_name = (
                    '"' + DATABASE_NAME.replace('"', '""') + '"'
                )

                conn.execute(
                    text(
                        f"CREATE DATABASE "
                        f"{quoted_database_name}"
                    )
                )

                print(
                    f"Database created: "
                    f"{DATABASE_NAME}"
                )

    finally:
        admin_engine.dispose()


ensure_database_exists()


# -----------------------------------------
# 5. Connect to existing or newly created DB
# -----------------------------------------
engine = create_engine(database_url)


# -----------------------------------------
# 6. Confirm database connection
# -----------------------------------------
with engine.connect() as conn:
    database_info = conn.execute(
        text("""
            SELECT
                current_database() AS database_name,
                current_user AS username,
                inet_server_addr() AS server_address,
                inet_server_port() AS server_port;
        """)
    ).mappings().one()


print("\nConnected to PostgreSQL")
print("Database:", database_info["database_name"])
print("User:", database_info["username"])
print("Server:", database_info["server_address"])
print("Port:", database_info["server_port"])


# -----------------------------------------
# 7. Read source tables
# -----------------------------------------
def read_table(table_name: str) -> pd.DataFrame:
    quoted_schema = SOURCE_SCHEMA.replace('"', '""')
    quoted_table = table_name.replace('"', '""')

    query = text(
        f'SELECT * FROM "{quoted_schema}"."{quoted_table}"'
    )

    return pd.read_sql(query, engine)


print("\nReading source tables from PostgreSQL...")

stops = read_table("stops")
routes = read_table("routes")
trips = read_table("trips")
stop_times = read_table("stop_times")
calendar = read_table("calendar")

print("Raw row counts")
print("stops:", len(stops))
print("routes:", len(routes))
print("trips:", len(trips))
print("stop_times:", len(stop_times))
print("calendar:", len(calendar))


# -----------------------------------------
# 8. Create clean schema if missing
# -----------------------------------------
quoted_clean_schema = CLEAN_SCHEMA.replace('"', '""')

with engine.begin() as conn:
    conn.execute(
        text(
            f'CREATE SCHEMA IF NOT EXISTS '
            f'"{quoted_clean_schema}"'
        )
    )

print(f"\nSchema ready: {CLEAN_SCHEMA}")


# -----------------------------------------
# 9. Example clean-table writing
# -----------------------------------------
# Add your cleaned DataFrames here after completing
# your cleaning and validation logic.
#
# Important:
# if_exists="fail" prevents overwriting existing tables.

clean_tables = {
    "stops_clean": stops,
    "routes_clean": routes,
    "trips_clean": trips,
    "stop_times_clean": stop_times,
    "calendar_clean": calendar,
}

print("\nWriting tables without overwriting existing tables...")

for table_name, dataframe in clean_tables.items():
    try:
        dataframe.to_sql(
            name=table_name,
            con=engine,
            schema=CLEAN_SCHEMA,
            if_exists="fail",
            index=False,
            method="multi",
            chunksize=10_000,
        )

        print(
            f"Created table "
            f"{CLEAN_SCHEMA}.{table_name}: "
            f"{len(dataframe):,} rows"
        )

    except ValueError as error:
        if "already exists" in str(error).lower():
            print(
                f"Skipped existing table: "
                f"{CLEAN_SCHEMA}.{table_name}"
            )
        else:
            raise


# -----------------------------------------
# 10. Verify clean tables
# -----------------------------------------
with engine.connect() as conn:
    table_list = conn.execute(
        text("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = :schema_name
            ORDER BY table_name;
        """),
        {"schema_name": CLEAN_SCHEMA},
    ).fetchall()


print("\nClean PostgreSQL tables:")
for row in table_list:
    print("-", row[0])

